In [1]:
import torch
import torchvision.ops as ops
import numpy as np
from tqdm import tqdm
from sklearn.metrics import average_precision_score
from scipy.optimize import linear_sum_assignment

In [3]:
import os
import json
from PIL import Image
import torchvision
import torch.nn.functional as F
import torchvision.transforms as T
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models import mobilenet_v3_large, mobilenet_v3_small
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
train_image_folder = "/content/drive/MyDrive/ThesisDataset/Splits/train"
val_image_folder = "/content/drive/MyDrive/ThesisDataset/Splits/val"
test_image_folder = "/content/drive/MyDrive/ThesisDataset/Splits/test"

train_annotation_file = "/content/drive/MyDrive/ThesisDataset/Splits/train.json"
val_annotation_file = "/content/drive/MyDrive/ThesisDataset/Splits/val.json"
test_annotation_file = "/content/drive/MyDrive/ThesisDataset/Splits/test.json"

In [5]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained = True)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:01<00:00, 136MB/s]


In [6]:
mobilenet_v3 = mobilenet_v3_large(pretrained=True)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth
100%|██████████| 21.1M/21.1M [00:00<00:00, 128MB/s]


In [7]:
backbone = mobilenet_v3.features

In [8]:
class MobileNetV3Backbone(nn.Module):
    def __init__(self, backbone, out_channels=256):
        super(MobileNetV3Backbone, self).__init__()
        self.backbone = backbone
        self.out_channels = out_channels
        self.conv = nn.Conv2d(960, self.out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.backbone(x)
        x = self.conv(x)
        return {"0": x}

custom_backbone = MobileNetV3Backbone(backbone=mobilenet_v3.features)

In [9]:
model = FasterRCNN(
    custom_backbone,
    num_classes=8,
)

In [10]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

model.transform = GeneralizedRCNNTransform(
    min_size=[224],
    max_size=224,
    image_mean=mean,
    image_std=std
)

In [11]:
anchor_sizes = ((32,),)
aspect_ratios = ((0.5, 1.0, 2.0),)
anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)
model.rpn.anchor_generator = anchor_generator

In [12]:
model

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=[224], max_size=224, mode='bilinear')
  )
  (backbone): MobileNetV3Backbone(
    (backbone): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps

In [13]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, json_file, image_dir, category_map=None):
        with open(json_file) as f:
            self.data = json.load(f)
        self.image_dir = image_dir

        self.indices = [img['id'] for img in self.data['images']]
        self.images = {img['id']: img['file_name'] for img in self.data['images']}

        self.annotations = {ann['image_id']: [] for ann in self.data['annotations']}

        if category_map is None:
            category_map = {
                1: 1,
                2: 2,
                5: 5,
                12: 3,
                14: 4,
                16: 6,
                19: 7
            }

        self.category_map = category_map

        for ann in self.data['annotations']:
            x, y, w, h = ann['bbox']
            x2, y2 = x + w, y + h
            mapped_category_id = self.category_map.get(ann['category_id'], ann['category_id'])
            self.annotations[ann['image_id']].append({
                'bbox': [x, y, x2, y2],
                'category_id': mapped_category_id
            })

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, indx):
        image_id = self.indices[indx]
        image_path = f"{self.image_dir}/{self.images[image_id]}"

        img = Image.open(image_path)

        mean = [0.5]
        std = [0.5]

        transform = T.Compose([
            T.Grayscale(num_output_channels=3),
            T.ToTensor(),
            T.Normalize(mean=mean * 3, std=std * 3)
            ])


        img = transform(img)

        annotations = self.annotations[image_id]

        boxes = [ann['bbox'] for ann in annotations]
        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)

        labels = torch.tensor([ann['category_id'] for ann in annotations], dtype=torch.int64)

        target = {'boxes': boxes_tensor, 'labels': labels}

        return img, target

def coco_collate_fn(batch):
    return batch


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [15]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0005
)

In [17]:
checkpoint_path = "/content/drive/MyDrive/ThesisTraining/model_epoch_15.pth"
checkpoint = torch.load(checkpoint_path, weights_only=True)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
model.eval()

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=[224], max_size=224, mode='bilinear')
  )
  (backbone): MobileNetV3Backbone(
    (backbone): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps

In [18]:
test_loader = torch.utils.data.DataLoader(
    CustomDataset(json_file=test_annotation_file, image_dir=test_image_folder),
    batch_size=32,
    shuffle=False,
    collate_fn=coco_collate_fn,
    pin_memory=True if device.type == 'cuda' else False
)

In [19]:
for batch in test_loader:
    images, targets = zip(*batch)
    print(f"Number of images in batch: {len(images)}")
    print(f"First image shape: {images[0].shape}")
    print(f"First target: {targets[0]}")
    break

Number of images in batch: 32
First image shape: torch.Size([3, 224, 224])
First target: {'boxes': tensor([[ 63.1542,  97.6916, 159.8590, 119.4009],
        [ 48.3524, 110.5198, 146.0441, 132.2291],
        [ 59.2071, 190.4493, 193.4097, 217.0925]]), 'labels': tensor([7, 7, 7])}


In [20]:
confidence_threshold = 0.3

all_preds = []
all_targets = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        images, targets = zip(*batch)
        images = [img.to(device) for img in images]

        outputs = model(images)

        for output, target in zip(outputs, targets):
            high_conf_ts = output['scores'] >= confidence_threshold

            filtered_output = {
                'boxes' : output['boxes'][high_conf_ts].cpu(),
                'labels' : output['labels'][high_conf_ts].cpu(),
                'scores' : output['scores'][high_conf_ts].cpu()
            }
            if filtered_output['boxes'].size(0) > 0:
                all_preds.append(filtered_output)
            all_targets.append({
                'boxes': target['boxes'].cpu(),
                'labels': target['labels'].cpu()
            })

Evaluating: 100%|██████████| 16/16 [05:12<00:00, 19.50s/it]


In [21]:
print(f"Length of predictions: {len(all_preds)}")
print(f"Length of targets: {len(all_targets)}")

Length of predictions: 64
Length of targets: 510


In [22]:
def calculate_iou(pred_boxes, gt_boxes):
    return ops.box_iou(pred_boxes, gt_boxes)

In [23]:
def calculate_precision_recall(all_preds, all_targets, iou_threshold=0.5):
    precisions = []
    recalls = []

    for preds, targets in zip(all_preds, all_targets):
        if len(preds['boxes']) == 0 or len(targets['boxes']) == 0:
            continue

        ious = calculate_iou(preds['boxes'], targets['boxes']).cpu().numpy()

        row_ind, col_ind = linear_sum_assignment(-ious)
        matched_ious = ious[row_ind, col_ind]

        valid_matches = matched_ious > iou_threshold
        tp = valid_matches.sum()

        precisions.append(tp / len(preds['boxes']))
        recalls.append(tp / len(targets['boxes']))

    precision = np.mean(precisions) if precisions else 0.0
    recall = np.mean(recalls) if recalls else 0.0
    return precision, recall


In [24]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
results = []

for thresh in thresholds:
    all_preds = []
    for output in outputs:
        high_conf_ts = output['scores'] >= thresh
        filtered_output = {
            'boxes': output['boxes'][high_conf_ts],
            'labels': output['labels'][high_conf_ts],
            'scores': output['scores'][high_conf_ts],
        }
        all_preds.append(filtered_output)

    precision, recall = calculate_precision_recall(all_preds, all_targets, iou_threshold=0.1)
    results.append((thresh, precision, recall))
    print(f"Threshold {thresh}: Precision {precision:.4f}, Recall {recall:.4f}")


Threshold 0.2: Precision 0.0227, Recall 0.1250
Threshold 0.3: Precision 0.0556, Recall 0.1667
Threshold 0.4: Precision 0.2500, Recall 0.5000
Threshold 0.5: Precision 0.0000, Recall 0.0000
Threshold 0.6: Precision 0.0000, Recall 0.0000
Threshold 0.7: Precision 0.0000, Recall 0.0000
Threshold 0.8: Precision 0.0000, Recall 0.0000
